# Atelier agent — sans puis avec mémoire

Suivi de la skill **openai-agents-sdk** installée dans ce projet (`.agents/skills/`).

**Objectif :** montrer la différence entre un agent **sans mémoire** (il oublie la question précédente) et un agent **avec session SQLite** (il s'en souvient).

- Runner **asynchrone** (`await Runner.run(...)`) comme recommandé par la skill dans un notebook ;
- clé API chargée depuis `.env` via `python-dotenv` — jamais codée en dur ;


In [1]:
# Étape 1 — Configuration (clé via .env, jamais affichée)
import os
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # trouve .env depuis le cwd
assert os.environ.get("GEMINI_API_KEY"), "GEMINI_API_KEY manquante ou vide dans .env"

from agents import Agent, Runner, SQLiteSession, set_tracing_disabled
from agents.extensions.models.litellm_provider import LitellmModel

# Pas de clé OpenAI : on désactive le tracing, comme conseillé par la doc SDK
set_tracing_disabled(True)

model = LitellmModel(model="gemini/gemini-flash-lite-latest")
print("SDK openai-agents prêt — modèle :", "gemini/gemini-flash-lite-latest")


SDK openai-agents prêt — modèle : gemini/gemini-flash-lite-latest


## Partie 1 — Agent **sans** mémoire

Même agent, deux tours de conversation **sans** session : le deuxième tour ne voit pas le premier.


In [2]:
# Étape 2 — Agent studio (sans mémoire) : première question
studio = Agent(
    name="Studio Assistant",
    instructions=(
        "Tu es l'assistant studio d'ATA SUARL, société de production vidéo/photo. "
        "Réponds à ses questions sur les services de l'entreprise : pré-production, "
        "tournage, montage/étalonnage, streaming live. Réponds en français en 2 phrases maximum."
    ),
    model=model,
)

question_1 = "Quels services propose ATA SUARL ?"
r1 = await Runner.run(studio, question_1)
print("Q1 :", question_1)
print("R1 :", r1.final_output)


Q1 : Quels services propose ATA SUARL ?
R1 : ATA SUARL est une société de production vidéo et photo qui vous accompagne de A à Z dans vos projets. Nos services couvrent la pré-production, le tournage, le montage, l'étalonnage ainsi que la diffusion en streaming live.


In [3]:
# Étape 3 — Question de suivi : l'agent a-t-il mémoire de Q1 ?
question_2 = "Quelle question je viens de te poser ?"
r2 = await Runner.run(studio, question_2)
print("Q2 :", question_2)
print("R2 :", r2.final_output)


Q2 : Quelle question je viens de te poser ?
R2 : Vous venez de me demander quelle question vous veniez de me poser. En tant qu'assistant d'ATA SUARL, je suis à votre disposition pour tout projet de pré-production, tournage, montage ou streaming live !


### Constat partie 1

→ Sans session, chaque appel part de zéro : R2 **ne reprend pas** la question précédente (« Quels services propose ATA SUARL ? ») — l'agent **oublie**.


## Partie 2 — Agent **avec** mémoire (SQLiteSession)

Même agent, mais chaque appel partage la même `SQLiteSession` : l'historique est persisté dans `agent_lab.db`.


In [4]:
# Étape 4 — Avec mémoire : première question (nouvelle session)
session = SQLiteSession("visite-1", "agent_lab.db")
m1 = await Runner.run(studio, question_1, session=session)
print("Q1 :", question_1)
print("R1 :", m1.final_output)


Q1 : Quels services propose ATA SUARL ?
R1 : ATA SUARL est une société de production vidéo et photo qui vous accompagne de A à Z dans vos projets. Nous couvrons l'ensemble de la chaîne de production, allant de la pré-production au tournage, en passant par le montage, l'étalonnage et le streaming live.


In [5]:
# Étape 5 — Question de suivi avec la même session : il doit se souvenir
m2 = await Runner.run(studio, question_2, session=session)
print("Q2 :", question_2)
print("R2 :", m2.final_output)


Q2 : Quelle question je viens de te poser ?
R2 : Tu viens de me demander "Quels services propose ATA SUARL ?". Je t'ai répondu en détaillant notre accompagnement complet en pré-production, tournage, montage, étalonnage et streaming live.


### Constat partie 2

→ Avec la session, R2 reprend la question précédente (les services d'ATA SUARL) : la mémoire fonctionne.

**Différence observée :** partie 1 = oublie · partie 2 = se souvient.
